In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
# ================================================
# MODEL 1 - Fixed & Stable Version
# Messy Mashup - From Scratch CNN
# Roll No: 24F1002246
# ================================================

import os
import numpy as np
import pandas as pd
import librosa
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import wandb
import warnings
warnings.filterwarnings("ignore")

# ====================== W&B - STABLE OFFLINE MODE ======================
os.environ["WANDB_SILENT"] = "true"
os.environ["WANDB_MODE"] = "offline"

wandb.init(
    project="DL-GenAi-t1-2026",
    name="Model1_StrongCNN_v2",
    mode="offline"
)
print("✅ W&B initialized in OFFLINE mode")

# ====================== Paths ======================
BASE_PATH = "/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup"
TRAIN_PATH = os.path.join(BASE_PATH, "genres_stems")
ESC50_PATH = os.path.join(BASE_PATH, "ESC-50-master/audio")

# ====================== Data Preparation ======================
genres = sorted(os.listdir(TRAIN_PATH))
label_map = {g: i for i, g in enumerate(genres)}
rev_label_map = {v: k for k, v in label_map.items()}

data = []
for genre in genres:
    genre_path = os.path.join(TRAIN_PATH, genre)
    for song in os.listdir(genre_path):
        song_path = os.path.join(genre_path, song)
        stems = {
            "drums": os.path.join(song_path, "drums.wav"),
            "bass": os.path.join(song_path, "bass.wav"),
            "vocals": os.path.join(song_path, "vocals.wav"),
            "other": os.path.join(song_path, "other.wav")
        }
        data.append((stems, label_map[genre]))

train_data, val_data = train_test_split(data, test_size=0.12, random_state=42, 
                                        stratify=[x[1] for x in data])

print(f"Train: {len(train_data)} | Val: {len(val_data)}")

# ====================== Dataset ======================
class MashupDataset(Dataset):
    def __init__(self, data, esc50_path=None, augment=True, target_len=22050*30):
        self.data = data
        self.target_len = target_len
        self.augment = augment
        self.esc50_files = [os.path.join(esc50_path, f) for f in os.listdir(esc50_path) if f.endswith(".wav")]
        
        self.class_data = {}
        for stems, label in data:
            if label not in self.class_data:
                self.class_data[label] = []
            self.class_data[label].append(stems)

    def fix_length(self, y):
        if len(y) < self.target_len:
            y = np.pad(y, (0, self.target_len - len(y)))
        else:
            y = y[:self.target_len]
        return y

    def load_audio(self, path):
        y, _ = librosa.load(path, sr=22050)
        return self.fix_length(y)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        stems, label = self.data[idx]
        audio_stems = []
        for key in ["drums", "bass", "vocals", "other"]:
            if self.augment and random.random() < 0.65:
                rand_stems = random.choice(self.class_data[label])
                path = rand_stems[key]
            else:
                path = stems[key]
            
            y = self.load_audio(path)
            if self.augment:
                y = y * random.uniform(0.6, 1.6)
                if random.random() < 0.25:
                    y = np.zeros_like(y)
            audio_stems.append(y)

        mix = np.sum(audio_stems, axis=0)

        if self.augment and random.random() < 0.5:
            rate = random.uniform(0.85, 1.18)
            mix = librosa.effects.time_stretch(mix, rate=rate)
            mix = self.fix_length(mix)

        if self.augment and self.esc50_files:
            for _ in range(random.randint(1, 3)):
                noise, _ = librosa.load(random.choice(self.esc50_files), sr=22050)
                start = random.randint(0, max(0, len(mix) - len(noise)))
                noise_pad = np.zeros_like(mix)
                end = min(len(mix), start + len(noise))
                noise_pad[start:end] = noise[:end-start] * random.uniform(0.25, 0.85)
                mix = mix + noise_pad

        mel = librosa.feature.melspectrogram(y=mix, sr=22050, n_mels=128, n_fft=2048, hop_length=512)
        mel = librosa.power_to_db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return torch.tensor(mel).unsqueeze(0).float(), label


train_loader = DataLoader(MashupDataset(train_data, ESC50_PATH, augment=True), 
                          batch_size=16, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(MashupDataset(val_data, ESC50_PATH, augment=False), 
                        batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

# ====================== Model ======================
class StemAwareCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(256, 512, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d((8, 8))
        )
        self.gru = nn.GRU(512, 256, num_layers=2, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(512, 10)

    def forward(self, x):
        x = self.features(x)
        x = x.flatten(2).permute(0, 2, 1)
        x, _ = self.gru(x)
        return self.fc(x.mean(1))


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = StemAwareCNN().to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
scaler = GradScaler()

# ====================== Training ======================
best_f1 = 0.0
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    for i, (x, y) in enumerate(train_loader):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        with autocast():
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    # Validation
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for x, y in val_loader:
            out = model(x.to(device))
            preds += torch.argmax(out, 1).cpu().tolist()
            trues += y.tolist()
    
    f1 = f1_score(trues, preds, average="macro")
    print(f"Epoch {epoch+1} | Val Macro F1: {f1:.4f}")
    wandb.log({"val_f1": f1})
    
    if f1 > best_f1:
        best_f1 = f1
        torch.save(model.state_dict(), "best_model1.pth")
        print(f"   >>> Best model saved! F1 = {best_f1:.4f}")

print(f"\n✅ Training Finished! Best Val Macro F1: {best_f1:.4f}")
wandb.finish()

# ====================== FIXED Test Dataset ======================
test_df = pd.read_csv(os.path.join(BASE_PATH, "test.csv"))

class TestDataset(Dataset):
    def __init__(self, df, target_len=22050*30):
        self.df = df
        self.target_len = target_len

    def fix_length(self, y):
        if len(y) < self.target_len:
            y = np.pad(y, (0, self.target_len - len(y)))
        else:
            y = y[:self.target_len]
        return y

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = os.path.join(BASE_PATH, self.df.iloc[idx]["filename"])
        y, _ = librosa.load(path, sr=22050)
        y = self.fix_length(y)
        mel = librosa.feature.melspectrogram(y=y, sr=22050, n_mels=128, n_fft=2048, hop_length=512)
        mel = librosa.power_to_db(mel)
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        return torch.tensor(mel).unsqueeze(0).float()

test_loader = DataLoader(TestDataset(test_df), batch_size=16, shuffle=False, num_workers=2)

model.load_state_dict(torch.load("best_model1.pth", map_location=device))
model.eval()

preds = []
with torch.no_grad():
    for x in test_loader:
        out = model(x.to(device))
        preds += torch.argmax(out, 1).cpu().tolist()

submission = pd.DataFrame({
    "id": test_df["id"],
    "genre": [rev_label_map[p] for p in preds]
})

submission.to_csv("submission_model1.csv", index=False)
print("✅ Submission saved as submission_model1.csv")
print(submission.head())

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

✅ W&B initialized in OFFLINE mode
Train: 880 | Val: 120
Epoch 1 | Val Macro F1: 0.0589
   >>> Best model saved! F1 = 0.0589
Epoch 2 | Val Macro F1: 0.0595
   >>> Best model saved! F1 = 0.0595
Epoch 3 | Val Macro F1: 0.2062
   >>> Best model saved! F1 = 0.2062
Epoch 4 | Val Macro F1: 0.2289
   >>> Best model saved! F1 = 0.2289
Epoch 5 | Val Macro F1: 0.2014
Epoch 6 | Val Macro F1: 0.2634
   >>> Best model saved! F1 = 0.2634
Epoch 7 | Val Macro F1: 0.3624
   >>> Best model saved! F1 = 0.3624
Epoch 8 | Val Macro F1: 0.2558
Epoch 9 | Val Macro F1: 0.3565
Epoch 10 | Val Macro F1: 0.3817
   >>> Best model saved! F1 = 0.3817

✅ Training Finished! Best Val Macro F1: 0.3817
✅ Submission saved as submission_model1.csv
   id    genre
0   1      pop
1   2     jazz
2   3   reggae
3   4  country
4   5  country


In [3]:
# # =========================
# # INSTALL
# # =========================
# !pip install transformers -q

# # =========================
# # IMPORTS
# # =========================
# import torch
# import torch.nn as nn
# import librosa
# import numpy as np
# import os
# import pandas as pd

# from torch.utils.data import Dataset, DataLoader
# from transformers import HubertModel
# from sklearn.metrics import f1_score

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # =========================
# # DATASET (TRAIN)
# # =========================
# class HubertDataset(Dataset):
    
#     def __init__(self, data, target_len=16000*15):  # 15 sec
#         self.data = data
#         self.target_len = target_len
    
#     def fix_length(self, y):
#         if len(y) < self.target_len:
#             y = np.pad(y, (0, self.target_len - len(y)))
#         else:
#             start = np.random.randint(0, len(y) - self.target_len)
#             y = y[start:start+self.target_len]
#         return y
    
#     def __len__(self):
#         return len(self.data)
    
#     def __getitem__(self, idx):
#         stems, label = self.data[idx]
        
#         audio = []
#         for s in stems.values():
#             y, _ = librosa.load(s, sr=16000)
#             y = self.fix_length(y)
#             audio.append(y)
        
#         mix = np.sum(audio, axis=0)
#         return torch.tensor(mix).float(), label


# # =========================
# # MODEL
# # =========================
# class HubertClassifier(nn.Module):
    
#     def __init__(self, num_classes=10):
#         super().__init__()
#         self.hubert = HubertModel.from_pretrained("facebook/hubert-base-ls960")
        
#         # FREEZE HUBERT
#         for param in self.hubert.parameters():
#             param.requires_grad = False
        
#         self.fc = nn.Sequential(
#             nn.Linear(self.hubert.config.hidden_size, 256),
#             nn.ReLU(),
#             nn.Dropout(0.3),
#             nn.Linear(256, num_classes)
#         )
    
#     def forward(self, x):
#         with torch.no_grad():
#             x = self.hubert(x).last_hidden_state
        
#         x = x.mean(dim=1)
#         return self.fc(x)


# # =========================
# # LOADERS
# # =========================
# train_loader = DataLoader(HubertDataset(train_data), batch_size=4, shuffle=True)
# val_loader = DataLoader(HubertDataset(val_data), batch_size=4)


# # =========================
# # TRAINING SETUP
# # =========================
# model = HubertClassifier(len(label_map)).to(device)

# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)

# EPOCHS = 5
# best_f1 = 0


# # =========================
# # TRAIN LOOP
# # =========================
# for epoch in range(EPOCHS):
    
#     model.train()
    
#     for x, y in train_loader:
#         x = x.to(device)
#         y = y.to(device)
        
#         optimizer.zero_grad()
#         out = model(x)
#         loss = criterion(out, y)
#         loss.backward()
#         optimizer.step()
    
#     # VALIDATION
#     model.eval()
#     preds, targets = [], []
    
#     with torch.no_grad():
#         for x, y in val_loader:
#             x = x.to(device)
#             out = model(x)
#             p = torch.argmax(out, 1).cpu().numpy()
            
#             preds.extend(p)
#             targets.extend(y.numpy())
    
#     f1 = f1_score(targets, preds, average="macro")
#     print(f"Epoch {epoch+1} F1:", f1)

#     if f1 > best_f1:
#         best_f1 = f1
#         torch.save(model.state_dict(), "best_model_hubert.pth")


# # =========================
# # TEST DATASET
# # =========================
# class TestDataset(Dataset):
    
#     def __init__(self, df, target_len=16000*15):
#         self.df = df
#         self.target_len = target_len
    
#     def fix_length(self, y):
#         if len(y) < self.target_len:
#             y = np.pad(y, (0, self.target_len - len(y)))
#         else:
#             y = y[:self.target_len]
#         return y
    
#     def __len__(self):
#         return len(self.df)
    
#     def __getitem__(self, idx):
        
#         path = BASE_PATH + "/" + self.df.iloc[idx]["filename"]
        
#         y, _ = librosa.load(path, sr=16000)
#         y = self.fix_length(y)
        
#         return torch.tensor(y).float()


# # =========================
# # TEST LOADER
# # =========================
# test_df = pd.read_csv(BASE_PATH + "/test.csv")

# test_loader = DataLoader(
#     TestDataset(test_df),
#     batch_size=4,
#     shuffle=False
# )


# # =========================
# # LOAD BEST MODEL
# # =========================
# model.load_state_dict(torch.load("best_model_hubert.pth", map_location=device))
# model.eval()


# # =========================
# # PREDICTION
# # =========================
# rev_label_map = {v:k for k,v in label_map.items()}

# predictions = []

# with torch.no_grad():
    
#     for x in test_loader:
        
#         x = x.to(device)
        
#         out = model(x)
        
#         pred = torch.argmax(out,1).cpu().numpy()
        
#         predictions.extend(pred)


# # =========================
# # SUBMISSION
# # =========================
# genres_pred = [rev_label_map[p] for p in predictions]

# submission = pd.DataFrame({
#     "id": test_df["id"],
#     "genre": genres_pred
# })

# submission.to_csv("submission.csv", index=False)

# print(submission.head())